In [0]:
%pip install "PyJWT[crypto]" requests azure-storage-blob

In [0]:
import json
import sys
from pathlib import Path

current = Path.cwd()

project_root = next(
    (
        path
        for path in (current, *current.parents)
        if (path / "config/banking.json").is_file()
        and (path / "src/openbanking").is_dir()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("No se ha encontrado la raíz del proyecto.")

source_path = str(project_root / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

from openbanking.enablebanking_client import EnableBankingClient
from openbanking.ingest_banking import ingest_banking
from openbanking.storage import Storage

config = json.loads(
    (project_root / "config/banking.json").read_text(encoding="utf-8")
)

private_key = dbutils.secrets.get(
    config["secret_scope"],
    config["private_key_secret"],
)

sessions = {
    bank: json.loads(
        dbutils.secrets.get(
            config["secret_scope"],
            bank_config["session_secret"],
        )
    )
    for bank, bank_config in config["banks"].items()
}

storage = Storage(
    account_name=config["storage"]["account_name"],
    account_key=dbutils.secrets.get("azure-storage", "graxiano"),
    container_name=config["storage"]["container_name"],
)

print("Configuración y clientes preparados.")

In [0]:
with EnableBankingClient(
    application_id=config["application_id"],
    private_key=private_key,
    base_url=config["base_url"],
    timeout=config["extraction"]["request_timeout_seconds"],
) as client:

    summary = ingest_banking(
        client=client,
        storage=storage,
        sessions=sessions,
        prefix=config["storage"]["prefix"],
        strategy=config["extraction"]["initial_strategy"],
    )

# Comprobar que el manifiesto se puede recuperar de ADLS.
saved_manifest = json.loads(
    storage.download(summary["manifest_path"])
)

if saved_manifest != summary:
    raise RuntimeError("El manifiesto recuperado no coincide.")

print("Estado:", summary["status"])
print("Manifiesto:", summary["manifest_path"])

for result in summary["results"]:
    print(
        result["bank"],
        "| endpoint:", result.get("endpoint", "session"),
        "| estado:", result["status"],
        "| registros:", result.get(
            "records_saved",
            result.get("record_count", 0),
        ),
        "| páginas:", result.get("pages_saved", 0),
    )

    if "error" in result:
        print("Error:", result["error"])

if summary["status"] != "success":
    raise RuntimeError(
        "Extracción incompleta. Consulta el manifiesto; "
        "las descargas correctas se han conservado."
    )

In [0]:
import json
from collections import defaultdict

manifest_path = (
    "personal/finanzas/openbanking/enablebanking/"
    "extraction_date=2026-11-02/"
    "run_id=20261102T012628943779Z_af43e26cf7cd/"
    "manifest.json"
)

manifest = json.loads(storage.download(manifest_path))

if manifest["status"] != "success":
    raise RuntimeError("La carga inicial no está completa.")

for result in manifest["results"]:
    if result.get("endpoint") != "transactions":
        continue

    dates = defaultdict(list)
    total = 0
    empty_pages = 0
    account_hashes = set()

    for path in result["files"]:
        document = json.loads(storage.download(path))
        transactions = document["response"]["transactions"]

        account_hash = document.get("identification_hash")
        if not account_hash:
            raise RuntimeError(
                "Falta identification_hash para identificar la cuenta."
            )

        account_hashes.add(account_hash)
        total += len(transactions)
        empty_pages += int(not transactions)

        for transaction in transactions:
            for field in (
                "booking_date",
                "transaction_date",
                "value_date",
            ):
                value = transaction.get(field)
                if value:
                    dates[field].append(value)

    if len(account_hashes) != 1:
        raise RuntimeError("Las páginas no pertenecen a la misma cuenta.")

    if total != result["records_saved"]:
        raise RuntimeError("El total no coincide con el manifiesto.")

    print(
        f"\nBanco: {result['bank']}"
        f" | Cuenta: {result['account_index']}"
    )
    print("Movimientos verificados:", total)
    print("Páginas vacías:", empty_pages)

    for field, values in sorted(dates.items()):
        print(f"{field}: {min(values)} → {max(values)}")

    if not dates:
        print("No hay fechas disponibles en los movimientos.")

In [0]:
import hashlib
import json

from azure.core.exceptions import ResourceExistsError

state_prefix = (
    config["storage"]["prefix"].rstrip("/")
    + "/_state/transactions"
)

for result in manifest["results"]:
    if result.get("endpoint") != "transactions":
        continue

    if result["status"] != "success":
        raise RuntimeError("No se puede inicializar una carga fallida.")

    first_page = json.loads(
        storage.download(result["files"][0])
    )

    identification_hash = first_page["identification_hash"]

    # Nombre de archivo estable sin incluir el identificador original.
    account_key = hashlib.sha256(
        identification_hash.encode("utf-8")
    ).hexdigest()

    state_path = (
        f"{state_prefix}/{result['bank']}/{account_key}/"
        f"{manifest['run_id']}.json"
    )

    state = {
        "schema_version": 1,
        "bank": result["bank"],
        "account_key": account_key,
        "run_id": manifest["run_id"],
        "mode": "initial",
        "covered_through": manifest["started_at"][:10],
        "completed_at": manifest["finished_at"],
        "manifest_path": manifest_path,
    }

    content = json.dumps(
        state,
        ensure_ascii=False,
        indent=2,
    ).encode("utf-8")

    try:
        storage.upload(state_path, content)
        action = "creado"

    except ResourceExistsError:
        existing = json.loads(storage.download(state_path))

        if existing != state:
            raise RuntimeError(
                "Ya existe un punto de control con contenido diferente."
            )

        action = "ya existente"

    print(f"{result['bank']}: punto de control {action}")

In [0]:
import importlib
import openbanking.storage as storage_module

importlib.reload(storage_module)

storage = storage_module.Storage(
    account_name=config["storage"]["account_name"],
    account_key=dbutils.secrets.get("azure-storage", "graxiano"),
    container_name=config["storage"]["container_name"],
)

print("Método disponible:", hasattr(storage, "list_paths"))

In [0]:
from datetime import date

from openbanking.checkpoints import CheckpointStore

checkpoints = CheckpointStore(
    storage=storage,
    prefix=config["storage"]["prefix"],
)

for bank, session in sessions.items():
    for index, account in enumerate(session["accounts"], start=1):
        plan = checkpoints.plan(
            bank=bank,
            identification_hash=account["identification_hash"],
            date_to=date(2026, 11, 3),
            overlap_days=config["extraction"]["overlap_days"],
        )

        assert plan["date_from"] == date(2026, 10, 26)
        assert plan["date_to"] == date(2026, 11, 3)

        print(
            bank,
            "| cuenta:", index,
            "| desde:", plan["date_from"],
            "| hasta:", plan["date_to"],
        )

In [0]:
from datetime import date

from openbanking.checkpoints import CheckpointStore
from openbanking.incremental_banking import run_incremental


class TestStorage:
    def __init__(self):
        self.files = {}

    def upload(self, path, content):
        if path in self.files:
            raise FileExistsError(path)
        self.files[path] = content

    def download(self, path):
        return self.files[path]

    def list_paths(self, prefix):
        return [
            path for path in self.files
            if path.startswith(prefix)
        ]


class TestClient:
    def get_balances(self, account_uid):
        return {"balances": []}

    def iter_transaction_pages(self, account_uid, **kwargs):
        assert kwargs["strategy"] == "default"
        assert kwargs["date_from"] == date(2026, 10, 26)
        assert kwargs["date_to"] == date(2026, 11, 3)

        yield {"transactions": [], "continuation_key": "next"}

        if account_uid == "fake-laboral":
            raise RuntimeError("Fallo simulado.")

        yield {
            "transactions": [{"entry_reference": "test-001"}],
            "continuation_key": None,
        }


class TestCheckpoints(CheckpointStore):
    def __init__(self, storage):
        super().__init__(storage, "tests")
        self.saved = []

    def plan(self, **kwargs):
        return {
            "strategy": "default",
            "date_from": date(2026, 10, 26),
            "date_to": date(2026, 11, 3),
        }

    def save(self, **kwargs):
        state = super().save(**kwargs)
        self.saved.append(state)
        return state


test_storage = TestStorage()
test_checkpoints = TestCheckpoints(test_storage)

outcome = run_incremental(
    client=TestClient(),
    storage=test_storage,
    checkpoints=test_checkpoints,
    sessions={
        "laboral": {
            "accounts": [{
                "uid": "fake-laboral",
                "identification_hash": "test-laboral",
            }]
        },
        "santander": {
            "accounts": [{
                "uid": "fake-santander",
                "identification_hash": "test-santander",
            }]
        },
    },
    prefix="tests",
    date_to=date(2026, 11, 3),
)

assert outcome["status"] == "partial_failure"

by_bank = {
    item["bank"]: item
    for item in outcome["accounts"]
}

assert not by_bank["laboral"]["checkpoint_saved"]
assert by_bank["santander"]["checkpoint_saved"]

assert len(test_checkpoints.saved) == 1
assert test_checkpoints.saved[0]["bank"] == "santander"

assert test_checkpoints.latest(
    "laboral", "test-laboral"
) is None

assert test_checkpoints.latest(
    "santander", "test-santander"
)["covered_through"] == "2026-11-03"

print("Prueba correcta.")
print("Laboral: fallo conservado sin avanzar el estado.")
print("Santander: extracción completada y estado actualizado.")